# Fig. — Clutter suppression ($\kappa$ sweep)

Interactive front-end for the stacked κ-sweep figure. Loader/summary helpers come
from `build_fig_clutter.py`; **`build_figure` is inlined below as an editable cell**
so you can override `ALGORITHM_STYLE`, colors, the κ\* marker, or panel layout
without editing the module. Re-run that cell to pick up your edits.

- **(top)** min-SINR vs κ — the communication cost of clutter avoidance.
- **(bottom)** sum-SCNR vs κ — the sensing benefit (the suppression); saturates
  once clutter is nulled.
- Dashed line marks the deployed operating point κ\* (= `ADMM_KAPPA`, 0.08).

**Compatible experiment:** `kappa_sweep` (`kind=='sweep'`).

In [ ]:
# Make the builder + cordis importable, then apply the paper rcParams.
import sys, logging
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = Path.cwd()
if str(FIG_DIR) not in sys.path:
    sys.path.insert(0, str(FIG_DIR))

import build_fig_clutter as B   # loader + helpers (single source of truth)
from cordis.plotting import apply_paper_style, figsize, plot_sweep, save_figure
from cordis.plotting.style import ALGORITHM_STYLE   # tweak here to restyle

USE_TEX = True   # set False on a node without pdflatex
apply_paper_style()
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

## 1. Load the κ-sweep run

`RESULT_DIR = None` auto-picks the newest run (an `array_<jobid>_aggregated/` is
preferred; the unreliable `latest` symlink is never used). Pin a path to re-render
a specific campaign — e.g. a stress-config run.

In [ ]:
RESULT_DIR = None   # e.g. 'results/exp_kappa_sweep/array_12345_aggregated'
result, result_dir = B.load_sweep(RESULT_DIR, experiment='kappa_sweep')
print('run:', result_dir)
print('κ grid:', sorted(result.sweep_results.keys()))

## 2. Resolve algorithms, SCNR metric, operating point

In [ ]:
KAPPA_STAR  = 0.08                         # deployed operating point (marked)
ONLY        = list(B.PREFERRED)            # Centralized / CORDIS-ADMM / CORDIS-Split
SCNR_METRIC = None                         # None -> first available of B.SCNR_METRIC_PREFERENCE

only = B._present_algorithms(result, ONLY)
scnr_metric = SCNR_METRIC or B._select_scnr_metric(result, only=only)
print('algorithms:', only)
print('scnr metric:', scnr_metric)

## 3. Numeric summary (caption sanity check)

Suppression gain = SCNR(κ_max) − SCNR(κ_min); comm cost = min-SINR(κ\*) − min-SINR(κ_min).

In [ ]:
summary = B.collect_summary(result, only, scnr_metric, KAPPA_STAR)
B._print_summary(result, summary, scnr_metric, KAPPA_STAR)

## 4. `build_figure` — editable copy

This is the **exact** function from `build_fig_clutter.py`, inlined so you can edit
it here (restyle, relabel, change the κ\* marker, swap panel order) and re-run. To
make an edit permanent, copy it back into the module. To restyle without editing the
body, mutate `ALGORITHM_STYLE` in the setup cell before calling it.

It depends only on the public `cordis.plotting` API plus `B.PREFERRED` /
`B._present_algorithms` / `B._select_scnr_metric` / `B.SINR_METRIC` / `B._SCNR_YLABEL`,
so it stays self-contained.

In [ ]:
# Bind the module-level names the function body references, so the inlined
# copy behaves identically to B.build_figure.
from typing import Optional, Sequence   # the inlined signature uses these
PREFERRED            = B.PREFERRED
SINR_METRIC          = B.SINR_METRIC
KAPPA_STAR_DEFAULT   = B.KAPPA_STAR_DEFAULT
_SCNR_YLABEL         = B._SCNR_YLABEL
_present_algorithms  = B._present_algorithms
_select_scnr_metric  = B._select_scnr_metric
logger               = logging.getLogger('fig_clutter.nb')

In [ ]:
def build_figure(result,
                 *,
                 only: Optional[Sequence[str]] = None,
                 scnr_metric: Optional[str] = None,
                 kappa_star: float = KAPPA_STAR_DEFAULT,
                 log_x: bool = False,
                 use_tex: bool = True):
    """Assemble the stacked κ-sweep figure and return ``(fig, only, scnr_metric)``.

    Built directly on ``cordis.plotting.plot_sweep`` (not scripts/_plot_common)
    so the paper builder stays decoupled from the scripts/ tree.  Reuses the
    project per-algorithm style for cross-figure consistency.
    """
    import matplotlib
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False
    import matplotlib.pyplot as plt
    from cordis.plotting import apply_paper_style, figsize, plot_sweep

    apply_paper_style()
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False

    only = list(only) if only else _present_algorithms(result, PREFERRED)
    if scnr_metric is None:
        scnr_metric = _select_scnr_metric(result, only=only)

    axis = result.sweep_axis
    if log_x and 0.0 in result.sweep_results:
        logger.warning("log_x requested but κ=0 is in the grid; using linear x.")
        log_x = False

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=figsize(width="single", aspect=3.5 / 2.6),
        sharex=True, gridspec_kw={"hspace": 0.12},
    )

    # (top) min-SINR vs κ  — communication cost
    plot_sweep(result.sweep_results, metric=SINR_METRIC, ax=ax_top,
               xlabel="", ylabel=r"min-SINR [dB]", only=only, log_x=log_x)
    ax_top.set_title(r"Clutter suppression vs penalty $\kappa$")

    # (bot) SCNR vs κ  — sensing benefit (the suppression)
    if scnr_metric is not None:
        plot_sweep(result.sweep_results, metric=scnr_metric, ax=ax_bot,
                   xlabel=axis.display or r"$\kappa$",
                   ylabel=_SCNR_YLABEL.get(scnr_metric, scnr_metric),
                   only=only, log_x=log_x)
    else:
        logger.warning("No SCNR metric available; bottom panel left empty.")
        ax_bot.set_xlabel(axis.display or r"$\kappa$")

    # operating-point marker κ* on both panels (label once, on top).
    # get_xaxis_transform() = (data-x, axes-y); tight_layout-safe.
    for ax in (ax_top, ax_bot):
        ax.axvline(kappa_star, ls="--", lw=0.9, color="0.45", zorder=0)
    ax_top.text(kappa_star, 0.96, r"$\kappa^\star$",
                transform=ax_top.get_xaxis_transform(),
                ha="left", va="top", fontsize="small", color="0.35")

    # one legend only (top); drop the duplicate on the bottom panel
    if ax_bot.get_legend() is not None:
        ax_bot.get_legend().remove()

    # NB: no fig.tight_layout() — it conflicts with the shared-x / hspace
    # stacked layout (same reason scripts/_plot_common.sweep_plot_pair omits
    # it).  save_figure() uses bbox_inches='tight', which handles the margins.
    return fig, only, scnr_metric


## 5. Render

In [ ]:
fig, used_only, used_scnr_metric = build_figure(
    result, only=only, scnr_metric=scnr_metric,
    kappa_star=KAPPA_STAR, use_tex=USE_TEX,
)
plt.show()

## 6. Save to `paper/figures/`

In [ ]:
out_stem = FIG_DIR.parents[1] / 'figures' / 'fig_clutter'
paths = save_figure(
    fig, out_stem, formats=('pdf',),
    metadata={
        'Figure': 'fig_clutter',
        'KappaStar': f'{KAPPA_STAR:g}',
        'Algorithms': ', '.join(used_only),
        'ScnrMetric': str(used_scnr_metric),
        'Run': result_dir.name,
    },
)
for p in paths:
    print('wrote', p)